# CaliDoc: complete-field reliability in action (WACV 2027)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MHHamdan/CaliDoc/blob/main/notebooks/CaliDoc_WACV2027_Demo.ipynb)

Token-level reliability is not complete-field reliability. CaliDoc calibrates the object a
downstream system consumes, the **complete field**. For every extracted field it reports
whether the field should **pass** through, go to **review**, or **abstain**.

This notebook runs from a clean runtime and uses public sources only:

- code from [GitHub](https://github.com/MHHamdan/CaliDoc);
- the released CORD checkpoint
  [`MHamdan/calidoc-layoutlmv3-large-cord`](https://huggingface.co/MHamdan/calidoc-layoutlmv3-large-cord);
- two real CORD v2 test receipts from
  [`MHamdan/calidoc-demo-examples`](https://huggingface.co/datasets/MHamdan/calidoc-demo-examples)
  (CC BY 4.0, Park et al. 2019).

It needs no Google Drive, Hugging Face login, or dataset credentials. A GPU runtime is
optional; on CPU each document takes a few seconds after the 1.4 GB model download.

**Calibration scope.** The guarantee bounds the expected per-document fraction of missed
gold fields. It holds for documents exchangeable with the checkpoint's calibration
documents. OCR is an external input, and the boundary-escape annotation audit is a
corpus-level diagnostic that is not run here.

## 1. Install CaliDoc from GitHub
The demo app lives in the repository, so we clone it and install the package from the clone.

In [ ]:
!git clone -q --depth 1 https://github.com/MHHamdan/CaliDoc.git CaliDoc
%pip install -q ./CaliDoc "gradio>=5.0"

## 2. Load the demo logic and the released CORD model
The notebook reuses the Gradio app's own functions (`demo/gradio/app.py`), so it shows exactly what the app shows. Inference is `calidoc.inference.CaliDocPredictor`.

In [ ]:
import sys
sys.path.insert(0, "CaliDoc/demo/gradio")
import app  # demo/gradio/app.py

import pandas as pd
import torch
from IPython.display import HTML, Markdown, display

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
cord = app.predictor(app.MODELS["CORD (receipts)"])   # downloads MHamdan/calidoc-layoutlmv3-large-cord
print("alpha =", cord.config["alpha"], "| pooled lambda =", round(cord.pooled_lambda, 4),
      "| field types with their own threshold:", len(cord.config["supported_field_types"]),
      "| pooled-fallback types:", len(cord.config["fallback_field_types"]))

## 3. Download the demo examples
Both receipts were selected by a fixed rule (median and 90th-percentile word count over the CORD test set), not by prediction quality.

In [ ]:
for name, ex in app.EXAMPLES.items():
    image, ocr, meta, expected = app.example_inputs(ex["doc_id"])
    print(f"{name}: {meta['document_id']} ({meta['split']}), {len(ocr['words'])} OCR words, "
          f"source {meta['source']} @ {meta['source_revision'][:8]}, license {meta['license']}")

In [ ]:
def show(name):
    vis, summary, rows, what, gt, result = app.run_example(name)
    display(Markdown(f"### {name}"))
    display(HTML(summary))
    display(vis)
    display(HTML(app.LEGEND_HTML))
    display(pd.DataFrame(rows, columns=app.TABLE_HEADERS).set_index("#"))
    display(Markdown(what))
    display(Markdown(gt))
    return result

## 4. Representative receipt: `cord_test_0042`

In [ ]:
result_0042 = show("CORD: Representative receipt")

Every extracted field here is correct, yet every field is routed for review. On CORD the
calibrated threshold of the supported type `MENU.SUB.CNT` is λ = 1.0, so that label enters
every word's prediction set. CaliDoc's sets are honest about this calibration regime
rather than hiding it, and the WACV paper reports CORD as its most expensive setting.

## 5. Review-routing case: `cord_test_0014`

In [ ]:
result_0014 = show("CORD: Review-routing case")

### Why review routing matters

One of the 25 extractions, **Coupon 100,000**, is wrong: the model predicted
`SUB_TOTAL.ETC`, while the CORD gold label is `TOTAL.TOTAL_ETC`. CaliDoc does not present
it as a confident final field. The field's worst word scores far above its threshold, so
its calibrated set **excludes the predicted label** (state `conflict`, action `review`).

The ground-truth check above is shown separately from routing: the routing itself uses no
labels. Raw statuses stay in the JSON:

In [ ]:
[{k: f[k] for k in ("type", "text", "set_sizes", "threshold", "threshold_source", "status")}
 for f in result_0014["fields"] if f["status"] != "ambiguous_review"]

## 6. Optional: launch the interactive app

This cell starts the same Gradio app (`demo/gradio/app.py`) with a temporary public link,
covering both the *Try a Real Example* and *Upload Your Own* tabs.

**The temporary Gradio link exists only while this Colab session is running.**

In [ ]:
demo = app.build_demo()
demo.launch(share=True)